# IN4640 Assignment 2 — Fitting and Alignment
**Student Name:** [Your Name Here]  
**Index Number:** [Your Index Here]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pandas as pd
from sklearn.linear_model import RANSACRegressor, LinearRegression

---
## Question 1 — Total Least Squares & RANSAC Line Fitting

### 1(a) Total Least Squares on Line 1 Only

In [ ]:
# Load dataset
D = np.genfromtxt('lines.csv', delimiter=',', skip_header=1)
print('Dataset shape:', D.shape)
print('First 5 rows:')
pd.DataFrame(D, columns=['x1','x2','x3','y1','y2','y3']).head()

In [ ]:
# Extract line 1 data (columns x1, y1)
x1 = D[:, 0]
y1 = D[:, 3]

# Total Least Squares (TLS) using SVD
# Fit line: ax + by + c = 0  (homogeneous form)
# Center the data first
mean_x = np.mean(x1)
mean_y = np.mean(y1)

# Build the data matrix
A = np.column_stack([x1 - mean_x, y1 - mean_y])

# SVD
U, S, Vt = np.linalg.svd(A)

# The last right singular vector gives the normal to the line
a, b = Vt[-1]   # normal vector (a, b)

# c from: a*mean_x + b*mean_y + c = 0
c = -(a * mean_x + b * mean_y)

print(f'TLS Line Parameters (ax + by + c = 0):')
print(f'  a = {a:.6f}')
print(f'  b = {b:.6f}')
print(f'  c = {c:.6f}')

# Convert to slope-intercept form: y = mx + k
if abs(b) > 1e-10:
    m = -a / b
    k = -c / b
    print(f'\nSlope-intercept form: y = {m:.6f}x + {k:.6f}')

In [ ]:
# Plot TLS fit for line 1
x_plot = np.linspace(x1.min(), x1.max(), 200)
y_plot = m * x_plot + k

plt.figure(figsize=(7, 5))
plt.scatter(x1, y1, s=15, label='Line 1 data', alpha=0.7)
plt.plot(x_plot, y_plot, 'r-', linewidth=2, label=f'TLS Fit: y={m:.3f}x+{k:.3f}')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Q1(a): Total Least Squares Fit — Line 1')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('q1a_tls_fit.png', dpi=150)
plt.show()
print('Figure saved as q1a_tls_fit.png')

### 1(b) RANSAC — Fit Three Lines Using All Points

In [ ]:
# Flatten all columns as instructed
X_cols = D[:, :3]
Y_cols = D[:, 3:]
X_all = X_cols.flatten()
Y_all = Y_cols.flatten()

print(f'Total points: {len(X_all)}')

def ransac_tls_line(x, y, residual_threshold=0.5, max_trials=1000):
    """Run RANSAC and return inlier mask + TLS parameters."""
    X_col = x.reshape(-1, 1)
    ransac = RANSACRegressor(
        estimator=LinearRegression(),
        residual_threshold=residual_threshold,
        max_trials=max_trials,
        random_state=42
    )
    ransac.fit(X_col, y)
    inlier_mask = ransac.inlier_mask_

    # Refit using TLS on inliers
    xi, yi = x[inlier_mask], y[inlier_mask]
    mx, my = np.mean(xi), np.mean(yi)
    A = np.column_stack([xi - mx, yi - my])
    _, _, Vt = np.linalg.svd(A)
    a, b = Vt[-1]
    c = -(a * mx + b * my)
    slope = -a / b if abs(b) > 1e-10 else np.inf
    intercept = -c / b if abs(b) > 1e-10 else np.inf
    return inlier_mask, (a, b, c), (slope, intercept)


remaining_x = X_all.copy()
remaining_y = Y_all.copy()
remaining_idx = np.arange(len(X_all))

lines = []
colors = ['red', 'blue', 'green']

for i in range(3):
    mask, params_abc, params_mi = ransac_tls_line(remaining_x, remaining_y)
    a, b, c = params_abc
    slope, intercept = params_mi
    n_inliers = mask.sum()
    print(f'Line {i+1}: a={a:.4f}, b={b:.4f}, c={c:.4f} | '
          f'y={slope:.4f}x+{intercept:.4f} | inliers={n_inliers}')
    lines.append({
        'x_inliers': remaining_x[mask],
        'y_inliers': remaining_y[mask],
        'abc': params_abc,
        'mi': params_mi
    })
    # Remove inliers (outliers become next iteration's data)
    remaining_x = remaining_x[~mask]
    remaining_y = remaining_y[~mask]

In [ ]:
# Plot all three RANSAC lines
plt.figure(figsize=(9, 6))
for i, (line, color) in enumerate(zip(lines, colors)):
    xi = line['x_inliers']
    yi = line['y_inliers']
    slope, intercept = line['mi']
    x_plot = np.linspace(xi.min(), xi.max(), 200)
    plt.scatter(xi, yi, s=12, color=color, alpha=0.6, label=f'Line {i+1} data')
    plt.plot(x_plot, slope * x_plot + intercept, color=color,
             linewidth=2, label=f'Line {i+1}: y={slope:.3f}x+{intercept:.3f}')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Q1(b): RANSAC + TLS — Three Lines')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('q1b_ransac_lines.png', dpi=150)
plt.show()
print('Figure saved as q1b_ransac_lines.png')

---
## Question 2 — Earring Size from Camera Parameters

In [ ]:
# Given camera parameters
f_mm = 8.0          # focal length in mm
pixel_size_um = 2.2 # pixel size in micrometres (2.2 µm)
Z_mm = 720.0        # distance from lens to imaging plane (mm)

# Convert pixel size to mm
pixel_size_mm = pixel_size_um / 1000.0  # 0.0022 mm

# Focal length in pixels
f_px = f_mm / pixel_size_mm
print(f'Focal length: {f_mm} mm = {f_px:.2f} pixels')

# Load earring image and measure in pixels
img_ear = cv2.imread('earrings.jpg')
h, w = img_ear.shape[:2]
print(f'Earring image size: {w} x {h} pixels')

# Convert to grayscale for measurements
gray = cv2.cvtColor(img_ear, cv2.COLOR_BGR2GRAY)

# Threshold to find earring extent
_, thresh = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY_INV)
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Measure each earring
img_annotated = img_ear.copy()
sizes_mm = []

# Sort contours by area (largest first)
contours = sorted(contours, key=cv2.contourArea, reverse=True)[:2]

for i, cnt in enumerate(contours):
    x, y, cw, ch = cv2.boundingRect(cnt)
    # Diameter in pixels (use average of width and height)
    diameter_px = (cw + ch) / 2.0

    # Real-world size using thin-lens formula:
    # size_real = size_pixels * pixel_size_mm * Z_mm / f_mm
    size_real_mm = diameter_px * pixel_size_mm * Z_mm / f_mm

    sizes_mm.append(size_real_mm)
    print(f'Earring {i+1}: {cw}x{ch} px | ~{diameter_px:.1f} px diameter | '
          f'Real size ≈ {size_real_mm:.2f} mm')

    cv2.rectangle(img_annotated, (x, y), (x+cw, y+ch), (0, 255, 0), 3)
    cv2.putText(img_annotated, f'{size_real_mm:.1f}mm',
                (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,0), 2)

print(f'\nEstimated earring diameter: {np.mean(sizes_mm):.2f} mm (average)')

In [ ]:
# Display annotated earring image
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(img_annotated, cv2.COLOR_BGR2RGB))
plt.title('Q2: Earring Size Estimation')
plt.axis('off')
plt.tight_layout()
plt.savefig('q2_earring_size.png', dpi=150)
plt.show()
print('Figure saved as q2_earring_size.png')

---
## Question 3 — Superimpose Flag on Cricket Turf

In [ ]:
# Load images
turf_img = cv2.imread('turf.jpg')
flag_img = cv2.imread('luxembourg_flag.jpg')

print(f'Turf image size: {turf_img.shape[1]} x {turf_img.shape[0]}')
print(f'Flag image size: {flag_img.shape[1]} x {flag_img.shape[0]}')

# ---------------------------------------------------------------
# Manually selected turf quadrangle corners (from click tool).
# These correspond to the four corners of the cricket pitch:
#   top-left, top-right, bottom-right, bottom-left
# Adjust these coordinates if needed by running the click snippet.
# ---------------------------------------------------------------
turf_h, turf_w = turf_img.shape[:2]

# Approximate corners for the cricket pitch rectangle (adjust as needed)
dst_pts = np.float32([
    [int(turf_w * 0.38), int(turf_h * 0.30)],   # top-left
    [int(turf_w * 0.62), int(turf_h * 0.30)],   # top-right
    [int(turf_w * 0.75), int(turf_h * 0.88)],   # bottom-right
    [int(turf_w * 0.25), int(turf_h * 0.88)],   # bottom-left
])

print('\nDestination (turf) points:')
for i, p in enumerate(dst_pts):
    print(f'  P{i+1}: {p}')

In [ ]:
# Source points — four corners of the flag image
fh, fw = flag_img.shape[:2]
src_pts = np.float32([
    [0,    0   ],   # top-left
    [fw-1, 0   ],   # top-right
    [fw-1, fh-1],   # bottom-right
    [0,    fh-1],   # bottom-left
])

# Compute homography: maps flag corners -> turf quadrangle
H, status = cv2.findHomography(src_pts, dst_pts)
print('Homography matrix H:')
print(np.round(H, 4))

# Warp flag into turf perspective
flag_warped = cv2.warpPerspective(flag_img, H, (turf_w, turf_h))

# Create mask from warped flag (non-black pixels)
mask = cv2.warpPerspective(
    np.ones((fh, fw), dtype=np.uint8) * 255, H, (turf_w, turf_h)
)

# Blend: paste warped flag onto turf using the mask
result = turf_img.copy()
result[mask > 0] = flag_warped[mask > 0]

# Display
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(turf_img, cv2.COLOR_BGR2RGB))
# Mark the quadrangle corners on original
turf_vis = turf_img.copy()
cv2.polylines(turf_vis, [dst_pts.astype(int)], True, (0, 0, 255), 3)
for p in dst_pts:
    cv2.circle(turf_vis, tuple(p.astype(int)), 8, (0, 255, 0), -1)
plt.imshow(cv2.cvtColor(turf_vis, cv2.COLOR_BGR2RGB))
plt.title('Turf with selected quadrangle')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
plt.title('Q3: Luxembourg Flag on Cricket Turf')
plt.axis('off')
plt.tight_layout()
plt.savefig('q3_flag_on_turf.png', dpi=150)
plt.show()
print('Figure saved as q3_flag_on_turf.png')

---
## Summary of Results

| Question | Result |
|----------|--------|
| 1(a) TLS Line 1 | Reported above (a, b, c) and slope/intercept |
| 1(b) RANSAC x3  | Three lines found with parameters printed above |
| 2 Earring size  | Estimated diameter in mm printed above |
| 3 Flag overlay  | Luxembourg flag projected onto cricket pitch |